In [1]:
!pip install ipympl
!pip install mplcursors

  Using cached ipympl-0.9.3-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached ipython_genutils-0.2.0-py2.py3-none-any.whl.metadata (755 bytes)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipympl-0.9.3-py2.py3-none-any.whl (511 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached ipython_genutils-0.2.0-py2.py3-none-any.whl (26 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
  Using cached mplcursors-0.7.1-py3-none-any.whl.metadata (2.0 kB)
Using cached mplcursors-0.7.1-py3-none-any.whl (20 kB)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.table import Table
from scipy.ndimage import median_filter, gaussian_filter1d
from scipy.signal import find_peaks, medfilt, savgol_filter
import glob

In [432]:
def find_trace(data, degree=3, nbins=80):
    ndisp, nspat = data.shape
    x_trace = []
    y_trace = []

    edges = np.linspace(0, ndisp, nbins + 1, dtype=int)

    for start, stop in zip(edges[:-1], edges[1:]):
        block = data[start:stop, :]
        profile = np.nanmedian(block, axis=0)
        profile = profile - np.nanmedian(profile)
        profile = gaussian_filter1d(profile, 2.0)

        peak = int(np.nanargmax(profile))
        lo = max(0, peak - 5)
        hi = min(nspat, peak + 6)

        weights = np.clip(profile[lo:hi], 0, None)
        pixels = np.arange(lo, hi)

        if weights.sum() > 0:
            center = np.sum(pixels * weights) / weights.sum()
        else:
            center = peak

        x_trace.append(0.5 * (start + stop))
        y_trace.append(center)

    x_trace = np.asarray(x_trace)
    y_trace = np.asarray(y_trace)

    coeff = np.polyfit(x_trace, y_trace, degree)
    fitted = np.polyval(coeff, x_trace)
    residual = y_trace - fitted

    keep = np.abs(residual) < 3 * np.std(residual)
    coeff = np.polyfit(x_trace[keep], y_trace[keep], degree)
    return coeff, x_trace, y_trace



def estimate_sky(row, center, sky_inner, sky_outer):
    pix = np.arange(row.size)
    sky_mask = ((pix >= center - sky_outer) & (pix <= center - sky_inner)) | \
               ((pix >= center + sky_inner) & (pix <= center + sky_outer))
    return np.nanmedian(row[sky_mask])



def extract_optimal(data, trace_coeff, aperture=6, sky_inner=10, sky_outer=24,
                    gain=1.0, read_noise=5.0):
    ndisp, nspat = data.shape
    pixels_disp = np.arange(ndisp)
    centers = np.polyval(trace_coeff, pixels_disp)
    spatial = np.arange(nspat)

    flux = np.full(ndisp, np.nan)
    variance_out = np.full(ndisp, np.nan)
    sky = np.full(ndisp, np.nan)

    for i, center in enumerate(centers):
        row = data[i, :]
        sky_i = estimate_sky(row, center, sky_inner, sky_outer)
        sky[i] = sky_i

        aperture_mask = np.abs(spatial - center) <= aperture
        signal = row[aperture_mask] - sky_i

        variance = np.maximum(np.abs(signal) / gain + read_noise**2, 1.0)

        profile = np.clip(signal, 0, None)
        profile = gaussian_filter1d(profile, 1.0)
        if profile.sum() <= 0:
            profile = np.ones_like(profile)
        profile = profile / profile.sum()

        denom = np.sum(profile**2 / variance)
        if denom > 0:
            flux[i] = np.sum(profile * signal / variance) / denom
            variance_out[i] = 1.0 / denom

    return pixels_disp, flux, variance_out, sky, centers

def centroid_peaks(y, peaks, half_width=2):
    centers = []
    baseline = np.nanmedian(y)
    for p in peaks:
        lo = max(0, p - half_width)
        hi = min(len(y), p + half_width + 1)
        xx = np.arange(lo, hi, dtype=float)
        yy = np.clip(y[lo:hi] - baseline, 0, None)
        if yy.sum() > 0:
            centers.append(np.sum(xx * yy) / yy.sum())
        else:
            centers.append(float(p))
    return np.asarray(centers)


def match_lines(peak_centers, reference_lines, coeff, tolerance=3.0):
    predicted = np.polyval(coeff, peak_centers)
    matched_pix = []
    matched_wave = []

    for pix_i, wave_i in zip(peak_centers, predicted):
        j = np.argmin(np.abs(reference_lines - wave_i))
        if abs(reference_lines[j] - wave_i) < tolerance:
            matched_pix.append(pix_i)
            matched_wave.append(reference_lines[j])

    return np.asarray(matched_pix), np.asarray(matched_wave)


def normalize_spectrum(flux, window=151, polyorder=3):
    window = int(window) | 1
    window = min(window, len(flux) - (1 - len(flux) % 2))

    y = np.nan_to_num(flux, nan=np.nanmedian(flux))
    coarse = medfilt(y, kernel_size=window)

    if window > polyorder + 3:
        continuum = savgol_filter(coarse, window, polyorder)
    else:
        continuum = coarse

    continuum = np.where(np.abs(continuum) <= 0,
                         np.nanmedian(continuum),
                         continuum)
    return flux / continuum, continuum



he_lines = np.array([3888.65, 4471.48, 4713.15, 4921.93,
                     5015.68, 5875.62, 6678.15, 7065.19])

ne_lines = np.array([5400.56, 5852.49, 5881.90, 5944.83,
                     5975.53, 6029.99, 6074.34, 6096.16,
                     6143.06, 6163.59, 6217.28, 6266.50,
                     6304.79, 6334.43, 6382.99, 6402.25,
                     6506.53, 6532.88, 6598.95, 6678.28,
                     6717.04, 6929.47, 7032.41, 7173.94,
                     7245.17])


def escala_lineal(wave,flux):
    import numpy as np
    from scipy.interpolate import interp1d

    orden = np.argsort(wave)
    wave_scale_sorted = wave[orden]
    flux_sorted = flux[orden]

    dr_original = np.diff(wave_scale_sorted)
    dr_original = np.append(dr_original, dr_original[-1]) 

    flux_density = flux_sorted / dr_original

    lambda_min = wave_scale_sorted.min()
    lambda_max = wave_scale_sorted.max()
    num_puntos = len(wave_scale_sorted)
    wave_linear = np.linspace(lambda_min, lambda_max, num_puntos)

    dr_linear = wave_linear[1] - wave_linear[0]

    interpolador = interp1d(wave_scale_sorted, flux_density, kind='linear', bounds_error=False, fill_value="extrapolate")
    flux_density_linear = interpolador(wave_linear)

    flux_linear = flux_density_linear * dr_linear

    return wave_linear,flux_linear


In [160]:
testima = fits.open('ALyb100251.fits')
testima.info

<bound method HDUList.info of [<astropy.io.fits.hdu.image.PrimaryHDU object at 0x000002350618D760>, <astropy.io.fits.hdu.image.ImageHDU object at 0x00000235061C2310>]>

In [161]:
testima[0].header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                   32 / number of bits per data pixel                  
NAXIS   =                    0 / number of data axes                            
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT = '        '                                                            
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
BZERO   =           2147483648 / offset data range to that of unsigned long     
BSCALE  =                    1 / default scaling factor                         
DATE-OBS= '2015-02-11T02:30:57.137' / Start of exposure                         
DATE_OBS= '2015-02-11T02:30:57.137' / Start of exposure                         
ORIGIN  = 'NOTSA   '                                                            
OBSERVAT= 'LaPalma '                                                            
TELESCOP= 'NOT     '        

In [162]:
testima[1].header

XTENSION= 'IMAGE   '           / IMAGE extension                                
BITPIX  =                   32 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axes                            
NAXIS1  =                  400 / length of data axis 1                          
NAXIS2  =                 2102 / length of data axis 2                          
PCOUNT  =                    0 / required keyword; must = 0                     
GCOUNT  =                    1 / required keyword; must = 1                     
BZERO   =           2147483648 / offset data range to that of unsigned long     
BSCALE  =                    1 / default scaling factor                         
BUNIT   = 'count   '                                                            
CCDNAME = 'CCD8    '                                                            
CTYPE1  = 'X       '           / Coordinate type of 1st axis                    
CTYPE2  = 'Y       '        

In [435]:
o_header = testima[0].header

In [163]:
g=testima[1].header['GAIN']
r=testima[1].header['RDNOISE']
print(g)
print(r)

0.33
4.2


In [7]:
lista=glob.glob('*.fits')
lista

['ALyb100249.fits',
 'ALyb100250.fits',
 'ALyb100251.fits',
 'ALyb100423.fits',
 'ALyb100424.fits',
 'ALyb100425.fits',
 'ALyb100426.fits',
 'ALyb100427.fits',
 'ALyb100428.fits',
 'ALyb100429.fits',
 'ALyb100430.fits',
 'ALyb100505.fits',
 'ALyb100506.fits',
 'ALyb100507.fits',
 'ALyb100508.fits',
 'ALyb100509.fits',
 'ALyb100510.fits',
 'ALyb100511.fits',
 'ALyb100512.fits',
 'ALyb100513.fits',
 'ALyb100514.fits',
 'ALyb100515.fits']

In [8]:
for item in lista:
    spec=fits.open(item)
    print(item,spec[0].header['IMAGETYP'],spec[0].header['OBJECT'])
    spec.close()

ALyb100249.fits WAVE HeNe
ALyb100250.fits FLAT Halogen
ALyb100251.fits OBJECT SP0644+375
ALyb100423.fits FLAT Halogen 1 alfosc-calibs
ALyb100424.fits FLAT Halogen 2 alfosc-calibs
ALyb100425.fits FLAT Halogen 3 alfosc-calibs
ALyb100426.fits FLAT Halogen 4 alfosc-calibs
ALyb100427.fits FLAT Halogen 5 alfosc-calibs
ALyb100428.fits WAVE He alfosc-calibs
ALyb100429.fits WAVE Ne alfosc-calibs
ALyb100430.fits WAVE ThAr alfosc-calibs
ALyb100505.fits BIAS alfosc-calibs bias
ALyb100506.fits BIAS alfosc-calibs bias
ALyb100507.fits BIAS alfosc-calibs bias
ALyb100508.fits BIAS alfosc-calibs bias
ALyb100509.fits BIAS alfosc-calibs bias
ALyb100510.fits BIAS alfosc-calibs bias
ALyb100511.fits BIAS alfosc-calibs bias
ALyb100512.fits BIAS alfosc-calibs bias
ALyb100513.fits BIAS alfosc-calibs bias
ALyb100514.fits BIAS alfosc-calibs bias
ALyb100515.fits BIAS alfosc-calibs bias


In [155]:
def show_image(data, title='', pmin=5, pmax=99):
    %matplotlib qt
    vmin, vmax = np.nanpercentile(data, [pmin, pmax])
    plt.figure(figsize=(8, 6.5))
    plt.imshow(data, aspect="auto", cmap="gray", vmin=vmin, vmax=vmax)
    plt.colorbar(label="ADU")
    plt.title(title)
    plt.xlabel("pixel espacial")
    plt.ylabel("pixel dispersion")
    plt.tight_layout()

In [448]:
#llamado a la función de mostrar gráfico:
data=fits.getdata('ALyb100251.fits')
show_image(data, title='', pmin=5, pmax=99)

In [11]:
groups={'BIAS': [], 'FLAT': [], 'WAVE': [], 'OBJECT': []}
for item in lista:
    header = fits.getheader(item)
    tipo=header['IMAGETYP']
    groups.setdefault(tipo, []).append(item)

In [12]:
groups

{'BIAS': ['ALyb100505.fits',
  'ALyb100506.fits',
  'ALyb100507.fits',
  'ALyb100508.fits',
  'ALyb100509.fits',
  'ALyb100510.fits',
  'ALyb100511.fits',
  'ALyb100512.fits',
  'ALyb100513.fits',
  'ALyb100514.fits',
  'ALyb100515.fits'],
 'FLAT': ['ALyb100250.fits',
  'ALyb100423.fits',
  'ALyb100424.fits',
  'ALyb100425.fits',
  'ALyb100426.fits',
  'ALyb100427.fits'],
 'WAVE': ['ALyb100249.fits',
  'ALyb100428.fits',
  'ALyb100429.fits',
  'ALyb100430.fits'],
 'OBJECT': ['ALyb100251.fits']}

In [13]:
bias_files=groups['BIAS']
bias_files

['ALyb100505.fits',
 'ALyb100506.fits',
 'ALyb100507.fits',
 'ALyb100508.fits',
 'ALyb100509.fits',
 'ALyb100510.fits',
 'ALyb100511.fits',
 'ALyb100512.fits',
 'ALyb100513.fits',
 'ALyb100514.fits',
 'ALyb100515.fits']

In [23]:
bias_stack=np.stack([fits.getdata(item) for item in bias_files])
clipped=sigma_clip(bias_stack, sigma=4.0, axis=0, masked=True)
master_bias2=np.ma.median(clipped, axis=0).filled(np.nan)

In [24]:
mb = fits.PrimaryHDU(master_bias2)
mb.writeto('master_bias2.fits',overwrite=True)

In [449]:
data=fits.getdata('master_bias2.fits')
show_image(data, title='', pmin=5, pmax=99)

In [26]:
flat_files = groups["FLAT"]
flat_files

['ALyb100250.fits',
 'ALyb100423.fits',
 'ALyb100424.fits',
 'ALyb100425.fits',
 'ALyb100426.fits',
 'ALyb100427.fits']

In [28]:
flat_stack=[]
for item in flat_files:
    data = fits.getdata(item)
    flat_stack.append(data - master_bias2)
flat_stack

[array([[ 2.6336e+04,  2.6337e+04,  2.6479e+04, ...,  1.9161e+04,
          1.9092e+04,  1.8920e+04],
        [ 2.4612e+04,  2.4182e+04,  2.4812e+04, ...,  1.6903e+04,
          1.6544e+04,  1.6453e+04],
        [ 2.2467e+04,  2.3212e+04,  2.3292e+04, ...,  1.5774e+04,
          1.5233e+04,  1.5563e+04],
        ...,
        [-5.5000e+01, -1.8000e+01, -2.4000e+01, ..., -2.7000e+01,
         -7.0000e+00, -6.0000e+00],
        [-4.9000e+01, -4.0000e+00, -2.0000e+01, ..., -1.0000e+01,
         -2.6000e+01, -3.2000e+01],
        [-7.2000e+01, -3.7000e+01, -2.7000e+01, ..., -8.0000e+00,
         -2.6000e+01, -4.0000e+00]]),
 array([[ 2.9665e+04,  2.9641e+04,  2.9944e+04, ...,  2.2443e+04,
          2.1689e+04,  2.1970e+04],
        [ 2.7687e+04,  2.7438e+04,  2.8582e+04, ...,  1.9527e+04,
          2.0259e+04,  1.9869e+04],
        [ 2.7012e+04,  2.7408e+04,  2.7114e+04, ...,  1.8067e+04,
          1.8438e+04,  1.8308e+04],
        ...,
        [-4.1000e+01, -4.2000e+01, -2.2000e+01, ..., -

In [29]:
flat_clipped = sigma_clip(flat_stack, sigma=4.0, axis=0, masked=True)
flat_combined = np.ma.median(flat_clipped, axis=0).filled(np.nan)

In [30]:
flat_combined

array([[ 2.85600e+04,  2.92700e+04,  2.94900e+04, ...,  2.23085e+04,
         2.17745e+04,  2.19140e+04],
       [ 2.73375e+04,  2.70550e+04,  2.74865e+04, ...,  1.92275e+04,
         1.92460e+04,  1.92140e+04],
       [ 2.61130e+04,  2.67410e+04,  2.65070e+04, ...,  1.75195e+04,
         1.79040e+04,  1.82010e+04],
       ...,
       [-5.30000e+01, -1.65000e+01, -1.30000e+01, ..., -2.15000e+01,
        -8.00000e+00, -1.55000e+01],
       [-4.15000e+01, -9.50000e+00, -1.60000e+01, ..., -1.80000e+01,
        -2.50000e+01, -2.70000e+01],
       [-5.90000e+01, -2.30000e+01, -1.85000e+01, ..., -1.95000e+01,
        -1.75000e+01, -1.05000e+01]])

In [33]:
data=flat_combined
show_image(data, title='', pmin=5, pmax=99)

In [34]:
fondo=median_filter(flat_combined,size=(100,100),mode="nearest")

In [36]:
data=fondo
show_image(data, title='', pmin=5, pmax=99)

In [37]:
fondo=fondo/np.average(fondo)

In [38]:
np.average(fondo)

0.9999999999999998

In [39]:
#eliminar valores negativos y ceros
fondo[fondo==0]=np.median(fondo)
fondo[fondo<0]=np.median(fondo)

In [40]:
flat_combined_sm=flat_combined/fondo
master_flat = flat_combined_sm/np.average(flat_combined_sm)
master_flat

array([[ 1.15399622e+00,  1.18772029e+00,  1.19916483e+00, ...,
         1.04291748e+00,  1.01795310e+00,  1.02447469e+00],
       [ 1.11163678e+00,  1.10014935e+00,  1.12742873e+00, ...,
         8.98881407e-01,  8.99746278e-01,  8.98250285e-01],
       [ 1.06385014e+00,  1.09685015e+00,  1.10488041e+00, ...,
         8.19032782e-01,  8.37008073e-01,  8.50892758e-01],
       ...,
       [-2.56174560e-03, -7.97524575e-04, -6.28352695e-04, ...,
        -1.03919869e-03, -3.86678582e-04, -7.49189752e-04],
       [-2.00589514e-03, -4.59180816e-04, -7.73357164e-04, ...,
        -8.70026809e-04, -1.20837057e-03, -1.30504021e-03],
       [-2.85175454e-03, -1.11170092e-03, -8.94194220e-04, ...,
        -9.42529043e-04, -8.45859398e-04, -5.07515639e-04]])

In [41]:
master_flat[master_flat==0]=np.median(master_flat)
master_flat[master_flat<0]=np.median(master_flat)

In [450]:
data=master_flat
show_image(data, title='', pmin=5, pmax=99)

In [43]:
mf = fits.PrimaryHDU(master_flat)
mf.writeto('master_flat.fits',overwrite=True)

In [44]:
oheader=fits.getheader(groups['OBJECT'][0])
odata=fits.getdata(groups['OBJECT'][0])

o_bias=odata-master_bias2
o_bias_flat=o_bias/master_flat

In [158]:
data=o_bias_flat
show_image(data, title='', pmin=5, pmax=99)

In [50]:
obf = fits.PrimaryHDU(o_bias_flat)
obf.writeto('objeto_b_f.fits',overwrite=True)

In [52]:
arc_files=groups['WAVE']
arc_files

['ALyb100249.fits', 'ALyb100428.fits', 'ALyb100429.fits', 'ALyb100430.fits']

In [53]:
for item in arc_files:
    adata=fits.getdata(item)
    a_bias=adata-master_bias2
    a_bias_flat=a_bias/master_flat
    fits.writeto(item[:-5]+'_b_f.fits',a_bias_flat.astype("float32"),overwrite=True)

In [55]:
trace_coeff, x_trace, y_trace = find_trace(o_bias_flat)

In [56]:
trace_coeff

array([ 1.94246306e-08, -5.67118731e-05,  4.07922079e-02,  1.94813496e+02])

In [156]:
vmin, vmax = np.nanpercentile(o_bias_flat, [5, 99.5])
plt.imshow(o_bias_flat, aspect="auto", cmap="gray", vmin=vmin, vmax=vmax)
y = np.arange(o_bias_flat.shape[0])
x = np.polyval(trace_coeff, y)
plt.plot(x, y, color="red", lw=1)
plt.show()

In [165]:
o_pix, o_flux, variance, sky, centers = extract_optimal(o_bias_flat, trace_coeff, aperture=10, sky_inner=20, 
                sky_outer=30, gain=g, read_noise=r)

In [168]:
o_pix

array([   0,    1,    2, ..., 2099, 2100, 2101])

In [167]:
o_flux

array([13046.35230922, 12736.84547797, 11825.0567826 , ...,
          88.98152058,    35.8460398 ,    21.13847561])

In [457]:
#ESPECTRO DEL OBJETO
plt.figure(figsize=(10, 6))
plt.plot(o_pix, o_flux, lw=0.8)
plt.xlabel("Pixel de dispersión")
plt.ylabel("Flujo")
plt.title("Extracción óptima")
plt.show()

In [76]:
arc_spectra = {}
arcs=glob.glob("A*_b_f.fits")
arcs

['ALyb100249_b_f.fits',
 'ALyb100428_b_f.fits',
 'ALyb100429_b_f.fits',
 'ALyb100430_b_f.fits']

In [86]:
ff1 = fits.open('ALyb100249.fits')
ob=ff1[0].header['OBJECT']
ob

'HeNe'

In [214]:
adata1=fits.getdata('ALyb100249_b_f.fits')
HeNe_pix, HeNe_flux, variance, sky, centers = extract_optimal(adata1, trace_coeff, aperture
        =8,gain=g,read_noise=r)
arc_spectra['ALyb100249_b_f.fits']=HeNe_flux
plt.plot(HeNe_pix,HeNe_flux,lw=0.8)
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title("HeNe")
plt.show()

In [92]:
ff1 = fits.open('ALyb100428.fits')
ob=ff1[0].header['OBJECT']
ob

'He alfosc-calibs'

In [461]:
adata2=fits.getdata('ALyb100428_b_f.fits')
He_pix, He_flux, variance, sky, centers = extract_optimal(adata2, trace_coeff, aperture
        =8,gain=g,read_noise=r)
arc_spectra['ALyb100428_b_f.fits']=He_flux
plt.figure(figsize=(8, 6))
plt.plot(He_pix,He_flux,lw=0.8)
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title("Arco de He")
plt.show()

In [94]:
ff1 = fits.open('ALyb100429.fits')
ob=ff1[0].header['OBJECT']
ob

'Ne alfosc-calibs'

In [462]:
adata3=fits.getdata('ALyb100429_b_f.fits')
Ne_pix, Ne_flux, variance, sky, centers = extract_optimal(adata3, trace_coeff, aperture
        =8,gain=g,read_noise=r)
arc_spectra['ALyb100429_b_f.fits']=Ne_flux
plt.figure(figsize=(8, 6))
plt.plot(Ne_pix,Ne_flux,lw=0.8)
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title("Arco de Ne")
plt.show()

In [101]:
ff1 = fits.open('ALyb100430.fits')
ob=ff1[0].header['OBJECT']
ob

'ThAr alfosc-calibs'

In [217]:
adata4=fits.getdata('ALyb100430_b_f.fits')
ThAr_pix, ThAr_flux, variance, sky, centers = extract_optimal(adata4, trace_coeff, aperture
        =8,gain=g,read_noise=r)
arc_spectra['ALyb100430_b_f.fits']=ThAr_flux
plt.plot(ThAr_pix,ThAr_flux,lw=0.8)
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title("ThAr")
plt.show()

**Extracción de líneas del arco He:**

In [227]:
arc_name = "ALyb100428_b_f.fits"
arc_flux = arc_spectra[arc_name]

baseline = np.nanmedian(arc_flux)
scatter = 1.4826 * np.nanmedian(np.abs(arc_flux - baseline))

He_peaks, properties = find_peaks(arc_flux, prominence=5 * scatter, distance=8)

In [228]:
He_peaks

array([   9,   19,   40,   75,  109,  124,  158,  174,  184,  201,  219,
        245,  279,  302,  318,  395,  439,  479,  519,  530,  564,  631,
        752,  985, 1001, 1283, 1317, 1387, 1471, 1654, 1680, 2051],
      dtype=int64)

In [229]:
plt.figure(figsize=(12, 4))
plt.plot(o_pix, arc_flux, lw=0.8)
plt.plot(He_peaks, arc_flux[He_peaks], "rx")
plt.gca().invert_xaxis()
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title(f"Lineas detectadas en {arc_name}")
plt.plot()

[]

In [181]:
#recentrar las líneas encontradas
peak_centers = centroid_peaks(arc_flux, peaks)

In [350]:
hep=[563,631,752,1316,1388,1471,1653,1680]
hew=[7281.349,7065.188,6678.149,4921.929,4713.143,4471.477,3964.727,3888.646]
#hep=[563,631,752,1001,1282,1316,1388,1471,1653,1680]
#hew=[7281.349,7065.188,6678.149,5875.618,5015.675,4921.929,4713.143,4471.477,3964.727,3888.646]

In [340]:
len(hep)

9

In [341]:
len(hew)

9

In [351]:
he_peak_centers = centroid_peaks(arc_spectra["ALyb100428_b_f.fits"], hep)

In [343]:
he_peak_centers

array([ 563.82176346,  630.87501609,  751.8148698 , 1001.32785221,
       1317.16682702, 1387.46705952, 1471.03004151, 1652.75264516,
       1680.40771481])

In [352]:
degree = 3
he_wave_coeff = np.polyfit(he_peak_centers, hew, degree)
he_wave_fit = np.polyval(he_wave_coeff, he_peak_centers)
he_residual = hew - he_wave_fit
he_rms = np.sqrt(np.mean(he_residual**2))
he_rms

0.5179484483066249

In [353]:
plt.figure()
plt.axhline(0, color="k", lw=0.8)
plt.plot(hew, he_residual, "o")
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Residuo [Angstrom]")
plt.title(f"RMS = {he_rms:.3f} Angstrom")

Text(0.5, 1.0, 'RMS = 0.518 Angstrom')

calibración del espectro:

In [354]:
he_wave_coeff

array([ 1.37358901e-07, -2.28418776e-04, -3.08675845e+00,  9.06942776e+03])

In [355]:
he_wavelength = np.polyval(he_wave_coeff, o_pix)
he_wavelength

array([9069.42775701, 9066.34077028, 9063.25332754, ..., 2854.21848794,
       2851.98899208, 2849.7607701 ])

In [356]:
plt.figure(figsize=(12, 4))
plt.plot(he_wavelength, o_flux, '-k', lw=0.8)
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Flujo")
plt.title("Espectro calibrado en longitud de onda")

Text(0.5, 1.0, 'Espectro calibrado en longitud de onda')

**Extracción de líneas del arco Ne:**

In [357]:
arc_name = "ALyb100429_b_f.fits"
arc_flux = arc_spectra[arc_name]

baseline = np.nanmedian(arc_flux)
scatter = 1.4826 * np.nanmedian(np.abs(arc_flux - baseline))

Ne_peaks, properties = find_peaks(arc_flux, prominence=5 * scatter, distance=8)

In [282]:
Ne_peaks

array([  54,   86,  109,  149,  191,  218,  230,  252,  286,  358,  370,
        383,  416,  436,  482,  500,  516,  540,  575,  598,  637,  669,
        738,  749,  777,  797,  805,  838,  859,  870,  881,  896,  914,
        925,  943,  954,  974,  983, 1008], dtype=int64)

In [358]:
plt.figure(figsize=(12, 4))
plt.plot(o_pix, arc_flux, lw=0.8)
plt.plot(Ne_peaks, arc_flux[Ne_peaks], "rx")
plt.gca().invert_xaxis()
plt.xlabel("Pixel")
plt.ylabel("Flujo")
plt.title(f"Lineas detectadas en {arc_name}")
plt.plot()

[]

In [398]:
Ne_p=[191,252,499,638,837]
Ne_w=[8495.36,8300.326,7488.872,7032.4127,6402.246]
#Ne_p=[191,229,252,285,499,515,576,598,638,665,837]
#Ne_w=[8495.36,8377.6070,8300.326,8136.4060,7488.872,7438.899,7245.1670,7173.9390,
#        7032.4127,6929.468,6402.246]

In [372]:
len(Ne_p)

9

In [373]:
len(Ne_w)

9

In [399]:
Ne_peak_centers = centroid_peaks(arc_spectra["ALyb100429_b_f.fits"], Ne_p)

In [288]:
Ne_peak_centers

array([190.84482031, 230.03576804, 252.27128688, 285.41496695,
       499.74034192, 515.6715787 , 575.21319598, 597.68590122,
       637.62133114, 664.08859688, 837.15172578])

In [400]:
degree = 3
Ne_wave_coeff = np.polyfit(Ne_peak_centers, Ne_w, degree)
Ne_wave_fit = np.polyval(Ne_wave_coeff, Ne_peak_centers)
Ne_residual = Ne_w - Ne_wave_fit
Ne_rms = np.sqrt(np.mean(Ne_residual**2))
Ne_rms

0.47866480292249197

In [401]:
plt.figure()
plt.axhline(0, color="k", lw=0.8)
plt.plot(Ne_w, Ne_residual, "o")
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Residuo [Angstrom]")
plt.title(f"RMS = {Ne_rms:.3f} Angstrom")

Text(0.5, 1.0, 'RMS = 0.479 Angstrom')

calibración del espectro:

In [402]:
Ne_wave_coeff

array([ 8.01854139e-07, -1.16295678e-03, -2.76185177e+00,  9.05883268e+03])

In [403]:
Ne_wavelength = np.polyval(Ne_wave_coeff, o_pix)
Ne_wavelength

array([9058.83268093, 9056.06966701, 9053.30433198, ..., 5553.31738555,
       5556.27575766, 5559.24190721])

In [404]:
plt.figure(figsize=(12, 4))
plt.plot(Ne_wavelength, o_flux, '-k', lw=0.8)
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Flujo")
plt.title("Espectro calibrado en longitud de onda")

Text(0.5, 1.0, 'Espectro calibrado en longitud de onda')

CONCATENANDO LOS PUNTOS DE AMBOS ELEMENTOS:

In [421]:
all_hep=[563,631,1316,1388,1471,1653,1680]
all_hew=[7281.349,7065.188,4921.929,4713.143,4471.477,3964.727,3888.646]

all_Ne_p=[191,499,837]
all_Ne_w=[8495.36,7488.872,6402.246]

In [422]:
#extraer los centros de los picos para cada arco individualmente
all_he_peak_centers = centroid_peaks(arc_spectra["ALyb100428_b_f.fits"], all_hep)
all_ne_peak_centers = centroid_peaks(arc_spectra["ALyb100429_b_f.fits"], all_Ne_p)

#combinar los pixeles y las longitudes de onda en arreglos únicos
todos_los_pixeles = np.concatenate([all_he_peak_centers, all_ne_peak_centers])
todas_las_ondas = np.concatenate([all_hew, all_Ne_w])

#ajuste polinomial maestro
all_degree = 3
master_coeff = np.polyfit(todos_los_pixeles, todas_las_ondas, all_degree)
master_fit = np.polyval(master_coeff, todos_los_pixeles)
master_residual = todas_las_ondas - master_fit
master_rms = np.sqrt(np.mean(master_residual**2))
print(f"RMS Maestro Combinado: {master_rms:.3f} Angstroms")

master_wavelength = np.polyval(master_coeff, o_pix)

RMS Maestro Combinado: 0.657 Angstroms


In [424]:
#COMPROBACIÓN
#calcular los residuos individuales usando el polinomio MAESTRO combinado
he_master_fit = np.polyval(master_coeff, all_he_peak_centers)
he_master_residual = all_hew - he_master_fit

ne_master_fit = np.polyval(master_coeff, all_ne_peak_centers)
ne_master_residual = all_Ne_w - ne_master_fit

#gráfica
plt.figure(figsize=(10, 5))
plt.axhline(0, color='r', linestyle='--', alpha=0.7)

# He
plt.plot(all_hew, he_master_residual, 'ob', label='Helio (Arco 28)', markersize=8)

# Ne
plt.plot(all_Ne_w, ne_master_residual, 'sr', label='Neón (Arco 29)', markersize=8)

plt.xlabel("Longitud de onda de laboratorio [Angstrom]")
plt.ylabel("Residuo (Laboratorio - Ajuste Maestro) [Angstrom]")
plt.title(f"Diagnóstico de Residuos (RMS Global Maestro = {master_rms:.2f} Å)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [427]:
#ESPECTRO CALIBRADO EL LONG. DE ONDA:
plt.figure(figsize=(12, 4))
plt.plot(master_wavelength, o_flux, '-k', lw=0.8)
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Flujo")
plt.title("Espectro calibrado en longitud de onda")

Text(0.5, 1.0, 'Espectro calibrado en longitud de onda')

In [428]:
#ESPECTRO CALIBRADO EL LONG. DE ONDA:
he_ne_wavelength = np.polyval(master_coeff, o_pix)

#máscara 
mask = (he_ne_wavelength > 3700) & (he_ne_wavelength < 9200)

plt.figure(figsize=(12, 5))
plt.plot(he_ne_wavelength[mask], o_flux[mask], '-k', lw=0.8)

#rotular las líneas 
lineas_balmer = {r'$H\alpha$': 6562.8, r'$H\beta$': 4861.3, r'$H\gamma$': 4340.5, r'$H\delta$': 4101.7}
for nombre, wl in lineas_balmer.items():
    if he_ne_wavelength[mask].min() < wl < he_ne_wavelength[mask].max():
        plt.axvline(wl, color='blue', linestyle=':', alpha=0.5)
        plt.text(wl + 20, plt.ylim()[1] * 0.9, nombre, color='blue', fontsize=10, weight='bold')

plt.xlabel("Longitud de onda [Angstrom]", fontsize=11)
plt.ylabel("Flujo (Cuentas / Respuesta)", fontsize=11)
plt.title(f"Espectro Completo Calibrado (Helio + Neón | RMS = {master_rms:.3f} Å)", fontsize=12, weight='bold')
plt.grid(True, alpha=0.25)
plt.show()

**LINEALIZAR:**

In [433]:
wl,fl = escala_lineal(master_wavelength, o_flux)

In [463]:
#ESPECTRO CALIBRADO EL LONG. DE ONDA LINEALIZADO:
plt.figure(figsize=(10, 6))
plt.plot(wl, fl, '-k', lw=0.8)
plt.xlabel("Longitud de onda [Angstrom]")
plt.ylabel("Flujo")
plt.title("Espectro calibrado y linealizado en longitud de onda")
plt.grid(True, alpha=0.25)

**GUARDAR EL ARCHIVO:**

In [444]:
spec_header=o_header.copy()
spec_header["CTYPE1"]="WAVE"
spec_header["CUNIT1"]="Angstrom"
spec_header["CRPIX1"]=1.0
spec_header["CRVAL1"]=float(wl[0])
spec_header["CDELT1"]=float(np.nanmedian(np.diff(wl)))

spec_header["CTYPE"]="WAVE"
spec_header["CUNIT"]="Angstrom"
spec_header["CRPIX"]=1.0
spec_header["CRVAL"]=float(wl[0])
spec_header["CDELT"]=float(np.nanmedian(np.diff(wl)))

spec_header["WCALRMS"]=float(master_rms)
spec_header["HISTORY"]="1DspectrumextractedinteractivelyinIPython."

In [445]:
spec_header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                   32 / number of bits per data pixel                  
NAXIS   =                    0 / number of data axes                            
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT = '        '                                                            
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
BZERO   =           2147483648 / offset data range to that of unsigned long     
BSCALE  =                    1 / default scaling factor                         
DATE-OBS= '2015-02-11T02:30:57.137' / Start of exposure                         
DATE_OBS= '2015-02-11T02:30:57.137' / Start of exposure                         
ORIGIN  = 'NOTSA   '                                                            
OBSERVAT= 'LaPalma '                                                            
TELESCOP= 'NOT     '        

In [443]:
np.diff(wl)

array([2.98876409, 2.98876409, 2.98876409, ..., 2.98876409, 2.98876409,
       2.98876409])

**Guardar el fichero:**

In [446]:
hdul=fits.PrimaryHDU(fl,header=spec_header)
hdul.writeto("object_spectrum_wave.fits",overwrite=True)

Número de cuentas del master bias y master flat:

In [447]:

bias_hdu = fits.open("master_bias2.fits")
flat_hdu = fits.open("master_flat.fits")

bias_data = bias_hdu[0].data
flat_data = flat_hdu[0].data

bias_mediana = np.median(bias_data)
flat_mediana = np.median(flat_data)

print(f"Número medio de cuentas del Master Bias: {bias_mediana:.2f} ADU")
print(f"Número medio de cuentas del Master Flat: {flat_mediana:.2f} ADU")

# cerrar los archivos FITS para liberar memoria
bias_hdu.close()
flat_hdu.close()

Número medio de cuentas del Master Bias: 10023.00 ADU
Número medio de cuentas del Master Flat: 1.02 ADU
